In [ ]:

import matplotlib.pyplot as plt
import scanpy as sc
import scvi
import sys
sys.path.append("../src/")
from multiHIVE.model import multiHIVE
import torch
import numpy as np

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)
torch.set_float32_matmul_precision("high")

In [ ]:
adata = sc.read_h5ad("../Data/RNA_ATAC/Brain-ISSAAC/Brain-ISSAAC-seq.h5ad")
adata.var_names_make_unique()
adata

In [ ]:
del adata.obsm
del adata.obsp

In [5]:
adata = scvi.data.organize_multiome_anndatas(adata)
adata = adata[:, adata.var["modality"].argsort()].copy()
sc.pp.filter_genes(adata, min_cells=int(adata.shape[0] * 0.01))
multiHIVE.setup_anndata(adata)
adata

/tmp/ipykernel_1938669/207618960.py:4: DeprecationWarning: multiHIVE is supposed to work with MuData. the use of anndata is deprecated and will be removed in scvi-tools 1.4. Please use setup_mudata
  multiHIVE.setup_anndata(adata)


AnnData object with n_obs × n_vars = 10361 × 174205
    obs: 'cell_type', 'batch', 'modality', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'modality', 'n_cells'
    uns: '_scvi_uuid', '_scvi_manager_uuid'

In [ ]:
vae = multiHIVE(adata, latent_distribution="normal",
                n_genes=(adata.var["modality"] == "Gene Expression").sum(),
                n_regions=(adata.var["modality"] == "Peaks").sum(),
                n_proteins=0,
                mi_loss = True,
               )

In [ ]:
vae.train()
vae.get_latent_representation()

In [ ]:
np.save("Brain-ISSAAC.npy", adata.obsm['Z_multiHIVE'])